In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV,cross_val_score
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.impute import KNNImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import IsolationForest


import category_encoders as ce
import warnings
warnings.filterwarnings("ignore")
data = pd.read_csv('/kaggle/input/predicting-estate-prices-in-tunisia/traindata.csv')
data.shape

(10198, 9)

In [16]:
data = pd.read_csv('/kaggle/input/predicting-estate-prices-in-tunisia/traindata.csv')
def preprocess_data(df):
    dff = df.copy()

    # Merge city and region
    dff['region'] = dff.apply(lambda x: x['city'] if x['region'] == 'Others' else x['region'], axis=1)
    
    # Create an indicator for size == -1 and replace -1 with NaN
    dff['size_is_negative_one'] = (dff['size'] == -1).astype(int)
    dff['size'] = dff['size'].replace(-1, np.nan)

    # Fill missing values using both forward and backward fill
    dff.fillna(method='bfill', inplace=True)
    dff.fillna(method='ffill', inplace=True)

    # Ensure no NaN values remain
    if dff.isnull().values.any():
        dff.fillna(dff.mean(), inplace=True)  # Fill any remaining NaNs with column means

    # Drop city column
    dff.drop(columns=['city'], inplace=True)

    # Mark outliers instead of removing them
    iso = IsolationForest(contamination=0.01, random_state=42)
    outlier_labels = iso.fit_predict(dff.select_dtypes(include=[np.number]))
    dff['outlier'] = (outlier_labels == -1).astype(int)

    # Rename log_price to price and drop the original column
    dff['price'] = dff['log_price']
    dff.drop('log_price', axis=1, inplace=True)

    # Replace -1 with 0 in bathroom_count and room_count
    dff[['bathroom_count', 'room_count']] = dff[['bathroom_count', 'room_count']].replace(-1, 0)

    # Apply pd.cut to create bins for size, room_count, and bathroom_count
    dff['size_bin'] = pd.cut(dff['size'], bins=[0, 50, 100, 150, 200, np.inf], labels=['XS', 'S', 'M', 'L', 'XL'])
    dff['room_count_bin'] = pd.cut(dff['room_count'], bins=[-1, 1, 3, 5, np.inf], labels=['Studio', 'Small', 'Medium', 'Large'])
    dff['bathroom_count_bin'] = pd.cut(dff['bathroom_count'], bins=[-1, 1, 2, np.inf], labels=['None', 'Few', 'Many'])

    # Feature engineering: interaction terms
    dff['size_room_interaction'] = dff['size'] * dff['room_count']
    dff['size_bathroom_interaction'] = dff['size'] * dff['bathroom_count']

    # Drop the indicator column for size == -1 if no longer needed
    dff.drop(columns=['size_is_negative_one'], inplace=True)

    # Encode categorical variables
    binary_encoder = ce.BinaryEncoder(cols=['type'])
    dff = binary_encoder.fit_transform(dff)

    # One-hot encode binned and categorical variables
    dff = pd.get_dummies(dff, columns=['category', 'region', 'size_bin', 'room_count_bin', 'bathroom_count_bin'], drop_first=True)

    # Split into features (X) and target (y)
    y = dff['price']
    X = dff.drop('price', axis=1)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Feature Scaling
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

    return X_train, X_test, y_train, y_test, scaler






In [17]:
X_train, X_test, y_train, y_test,scaler = preprocess_data(data)

In [18]:
X_train

,ID,room_count,bathroom_count,size,type_0,type_1,outlier,size_room_interaction,size_bathroom_interaction,category_Bureaux et Plateaux,...,region_Zéramdine,size_bin_S,size_bin_M,size_bin_L,size_bin_XL,room_count_bin_Small,room_count_bin_Medium,room_count_bin_Large,bathroom_count_bin_Few,bathroom_count_bin_Many
8161,1.304783,1.596932,1.037128,0.588993,1.255040,-1.255040,-0.104425,0.899683,0.540219,-0.198080,...,-0.011072,-0.726243,-0.594056,-0.355428,1.926178,-0.874227,2.697005,-0.19908,2.305264,-0.243604
8696,-1.716464,-1.076308,-1.094884,-0.270657,-0.796787,0.796787,-0.104425,-0.411410,-0.367039,-0.198080,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,-0.874227,-0.370782,-0.19908,-0.433790,-0.243604
6044,0.184268,-0.007012,-0.028878,-0.334884,1.255040,-1.255040,-0.104425,-0.213872,-0.196172,5.048466,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,1.143867,-0.370782,-0.19908,-0.433790,-0.243604
5419,-0.143846,-1.076308,-1.094884,-0.320063,-0.796787,0.796787,-0.104425,-0.411410,-0.367039,-0.198080,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,-0.874227,-0.370782,-0.19908,-0.433790,-0.243604
4196,-0.920815,0.527636,2.103133,0.124584,-0.796787,0.796787,-0.104425,0.128760,0.567436,-0.198080,...,-0.011072,-0.726243,-0.594056,-0.355428,1.926178,1.143867,-0.370782,-0.19908,-0.433790,4.105022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,-1.024231,0.527636,-0.028878,-0.399111,1.255040,-1.255040,-0.104425,-0.149191,-0.215829,-0.198080,...,-0.011072,1.376949,-0.594056,-0.355428,-0.519163,1.143867,-0.370782,-0.19908,-0.433790,-0.243604
5191,-1.389433,-1.076308,-1.094884,0.183871,-0.796787,0.796787,-0.104425,-0.411410,-0.367039,-0.198080,...,-0.011072,-0.726243,-0.594056,-0.355428,1.926178,-0.874227,-0.370782,-0.19908,-0.433790,-0.243604
5390,1.177815,-1.076308,-1.094884,-0.152085,-0.796787,0.796787,-0.104425,-0.411410,-0.367039,-0.198080,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,-0.874227,-0.370782,-0.19908,-0.433790,-0.243604
860,-1.419754,-0.007012,-0.028878,-0.596732,1.255040,-1.255040,-0.104425,-0.306523,-0.276313,-0.198080,...,-0.011072,1.376949,-0.594056,-0.355428,-0.519163,1.143867,-0.370782,-0.19908,-0.433790,-0.243604


In [19]:
X_train.head()

,ID,room_count,bathroom_count,size,type_0,type_1,outlier,size_room_interaction,size_bathroom_interaction,category_Bureaux et Plateaux,...,region_Zéramdine,size_bin_S,size_bin_M,size_bin_L,size_bin_XL,room_count_bin_Small,room_count_bin_Medium,room_count_bin_Large,bathroom_count_bin_Few,bathroom_count_bin_Many
8161,1.304783,1.596932,1.037128,0.588993,1.255040,-1.255040,-0.104425,0.899683,0.540219,-0.198080,...,-0.011072,-0.726243,-0.594056,-0.355428,1.926178,-0.874227,2.697005,-0.19908,2.305264,-0.243604
8696,-1.716464,-1.076308,-1.094884,-0.270657,-0.796787,0.796787,-0.104425,-0.411410,-0.367039,-0.198080,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,-0.874227,-0.370782,-0.19908,-0.433790,-0.243604
6044,0.184268,-0.007012,-0.028878,-0.334884,1.255040,-1.255040,-0.104425,-0.213872,-0.196172,5.048466,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,1.143867,-0.370782,-0.19908,-0.433790,-0.243604
5419,-0.143846,-1.076308,-1.094884,-0.320063,-0.796787,0.796787,-0.104425,-0.411410,-0.367039,-0.198080,...,-0.011072,-0.726243,1.683344,-0.355428,-0.519163,-0.874227,-0.370782,-0.19908,-0.433790,-0.243604
4196,-0.920815,0.527636,2.103133,0.124584,-0.796787,0.796787,-0.104425,0.128760,0.567436,-0.198080,...,-0.011072,-0.726243,-0.594056,-0.355428,1.926178,1.143867,-0.370782,-0.19908,-0.433790,4.105022


In [20]:
y_test

5133    2.875061
3742    3.060698
9482    5.301030
9041    2.380211
5135    2.653213
          ...   
5767    5.886491
4905    4.903090
5564    5.518514
2820    2.681241
4438    4.361728
Name: price, Length: 2040, dtype: float64

In [21]:
from sklearn.model_selection import cross_val_score

# Define Models
models = {
    "Linear Regression": LinearRegression(),
    #"Decision Tree": DecisionTreeRegressor(),
    #"Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    #"XGBoost": XGBRegressor(eval_metric='rmse', use_label_encoder=False),
    "CatBoost": CatBoostRegressor(verbose=0),
    #"Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=1),
    "MLP Regressor": MLPRegressor(
        hidden_layer_sizes=(100, 50),
        activation='relu',
        solver='adam',
        max_iter=500,
        random_state=1,
        early_stopping=True,  # Enable early stopping
        validation_fraction=0.1,  # Use 10% of the training data for validation
        n_iter_no_change=10  # Stop if no improvement after 10 iterations
    )
}

# Train and cross-validate each model
cv_results = {}

print("\nModel Training and Cross-Validation:")
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)
    rmse_scores = np.sqrt(-scores)  # Convert negative MSE to RMSE
    cv_results[name] = {
        'mean_rmse': rmse_scores.mean(),
        'std_rmse': rmse_scores.std()
    }
    model.fit(X_train, y_train)
    print(f"{name} trained. Mean CV RMSE: {rmse_scores.mean():.2f}, Std Dev: {rmse_scores.std():.2f}")

# Display sorted cross-validation results
print("\nCross-Validation Results (Sorted by Mean RMSE):")
sorted_cv_results = sorted(cv_results.items(), key=lambda x: x[1]['mean_rmse'])
for name, result in sorted_cv_results:
    print(f"{name} - Mean RMSE: {result['mean_rmse']:.2f}, Std Dev: {result['std_rmse']:.2f}")



Model Training and Cross-Validation:
Linear Regression trained. Mean CV RMSE: 17717984807850.58, Std Dev: 9516939188356.04
Gradient Boosting trained. Mean CV RMSE: 0.53, Std Dev: 0.02
CatBoost trained. Mean CV RMSE: 0.53, Std Dev: 0.02
MLP Regressor trained. Mean CV RMSE: 0.72, Std Dev: 0.02

Cross-Validation Results (Sorted by Mean RMSE):
CatBoost - Mean RMSE: 0.53, Std Dev: 0.02
Gradient Boosting - Mean RMSE: 0.53, Std Dev: 0.02
MLP Regressor - Mean RMSE: 0.72, Std Dev: 0.02
Linear Regression - Mean RMSE: 17717984807850.58, Std Dev: 9516939188356.04


In [22]:
# Define parameter grids
mlp_param_grid = {
    'hidden_layer_sizes': [(100, 50), ],
    'activation': ['relu'],
    'solver': ['adam'],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [500,]
}

gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

catboost_param_grid = {
    'iterations': [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'depth': [4, 6, 8]
}

# Perform grid search for each model
print("Performing Grid Search for MLP Regressor...")
mlp_grid = GridSearchCV(
    MLPRegressor(early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=1), 
    param_grid=mlp_param_grid, 
    scoring='neg_mean_squared_error', 
    cv=3, 
    n_jobs=-1
)
mlp_grid.fit(X_train, y_train)
print(f"Best parameters for MLP Regressor: {mlp_grid.best_params_}")

print("\nPerforming Grid Search for Gradient Boosting...")
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=1), 
    param_grid=gb_param_grid, 
    scoring='neg_mean_squared_error', 
    cv=3, 
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)
print(f"Best parameters for Gradient Boosting: {gb_grid.best_params_}")

print("\nPerforming Grid Search for CatBoost...")
catboost_grid = GridSearchCV(
    CatBoostRegressor(verbose=0, random_state=1), 
    param_grid=catboost_param_grid, 
    scoring='neg_mean_squared_error', 
    cv=3, 
    n_jobs=-1
)
catboost_grid.fit(X_train, y_train)
print(f"Best parameters for CatBoost: {catboost_grid.best_params_}")


Performing Grid Search for MLP Regressor...
Best parameters for MLP Regressor: {'activation': 'relu', 'hidden_layer_sizes': (100, 50), 'learning_rate': 'constant', 'max_iter': 500, 'solver': 'adam'}

Performing Grid Search for Gradient Boosting...
Best parameters for Gradient Boosting: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}

Performing Grid Search for CatBoost...
Best parameters for CatBoost: {'depth': 8, 'iterations': 200, 'learning_rate': 0.1}


In [23]:
# Evaluate the best models using cross-validation
models = {
    'MLP Regressor': mlp_grid.best_estimator_,
    'Gradient Boosting': gb_grid.best_estimator_,
    'CatBoost': catboost_grid.best_estimator_
}

print("\nCross-Validation Results:")
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)
    rmse_scores = np.sqrt(-scores)  # Convert negative MSE to RMSE
    cv_results[name] = {
        'mean_rmse': rmse_scores.mean(),
        'std_rmse': rmse_scores.std()
    }
    print(f"{name} - Mean CV RMSE: {rmse_scores.mean():.2f}, Std Dev: {rmse_scores.std():.2f}")

# Evaluate the best models on the test set
print("\nTest Set Evaluation:")
for name, model in models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name} - RMSE: {rmse:.2f}, R2: {r2:.2f}")



Cross-Validation Results:
MLP Regressor - Mean CV RMSE: 0.72, Std Dev: 0.02
Gradient Boosting - Mean CV RMSE: 0.53, Std Dev: 0.01
CatBoost - Mean CV RMSE: 0.53, Std Dev: 0.02

Test Set Evaluation:
MLP Regressor - RMSE: 0.66, R2: 0.78
Gradient Boosting - RMSE: 0.56, R2: 0.84
CatBoost - RMSE: 0.55, R2: 0.85


In [24]:
# Define models
catboost_best = catboost_grid.best_estimator_
gb_best = gb_grid.best_estimator_

# Create an ensemble using VotingRegressor
ensemble_model = VotingRegressor(estimators=[
    ('CatBoost', catboost_best),
    ('Gradient Boosting', gb_best)
])

# Fit the ensemble model
ensemble_model.fit(X_train, y_train)

# Evaluate the ensemble model using cross-validation
print("\nCross-Validation Results:")
models = {
    'Gradient Boosting': gb_best,
    'CatBoost': catboost_best,
    'Ensemble (CatBoost + Gradient Boosting)': ensemble_model
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)
    rmse_scores = np.sqrt(-scores)  # Convert negative MSE to RMSE
    cv_results[name] = {
        'mean_rmse': rmse_scores.mean(),
        'std_rmse': rmse_scores.std()
    }
    print(f"{name} - Mean CV RMSE: {rmse_scores.mean():.2f}, Std Dev: {rmse_scores.std():.2f}")

# Evaluate the ensemble model on the test set
print("\nTest Set Evaluation:")
for name, model in models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name} - RMSE: {rmse:.2f}, R2: {r2:.2f}")



Cross-Validation Results:
Gradient Boosting - Mean CV RMSE: 0.53, Std Dev: 0.01
CatBoost - Mean CV RMSE: 0.53, Std Dev: 0.02
Ensemble (CatBoost + Gradient Boosting) - Mean CV RMSE: 0.53, Std Dev: 0.02

Test Set Evaluation:
Gradient Boosting - RMSE: 0.56, R2: 0.84
CatBoost - RMSE: 0.55, R2: 0.85
Ensemble (CatBoost + Gradient Boosting) - RMSE: 0.55, R2: 0.84


In [27]:
def prepare_test_data(df, train_columns):
    # Merge city and region
    df['region'] = df.apply(lambda x: x['city'] if x['region'] == 'Others' else x['region'], axis=1)
    
    # Create an indicator for size == -1 and replace -1 with NaN
    df['size_is_negative_one'] = (df['size'] == -1).astype(int)
    df['size'] = df['size'].replace(-1, np.nan)

    # Fill missing values using both forward and backward fill
    df.fillna(method='bfill', inplace=True)
    df.fillna(method='ffill', inplace=True)

    # Ensure no NaN values remain
    if df.isnull().values.any():
        df.fillna(df.mean(), inplace=True)  # Fill any remaining NaNs with column means

    # Drop city column
    df.drop(columns=['city'], inplace=True)


    # Replace -1 with 0 in bathroom_count and room_count
    df[['bathroom_count', 'room_count']] = df[['bathroom_count', 'room_count']].replace(-1, 0)

    # Apply pd.cut to create bins for size, room_count, and bathroom_count
    df['size_bin'] = pd.cut(df['size'], bins=[0, 50, 100, 150, 200, np.inf], labels=['XS', 'S', 'M', 'L', 'XL'])
    df['room_count_bin'] = pd.cut(df['room_count'], bins=[-1, 1, 3, 5, np.inf], labels=['Studio', 'Small', 'Medium', 'Large'])
    df['bathroom_count_bin'] = pd.cut(df['bathroom_count'], bins=[-1, 1, 2, np.inf], labels=['None', 'Few', 'Many'])

    # Feature engineering: interaction terms
    df['size_room_interaction'] = df['size'] * df['room_count']
    df['size_bathroom_interaction'] = df['size'] * df['bathroom_count']

    # Drop the indicator column for size == -1 if no longer needed
    df.drop(columns=['size_is_negative_one'], inplace=True)
  # Mark outliers instead of removing them
    iso = IsolationForest(contamination=0.01, random_state=42)
    outlier_labels = iso.fit_predict(df.select_dtypes(include=[np.number]))
    df['outlier'] = (outlier_labels == -1).astype(int)
    # Encode categorical variables
    binary_encoder = ce.BinaryEncoder(cols=['type'])
    df = binary_encoder.fit_transform(df)

    # One-hot encode binned and categorical variables
    df = pd.get_dummies(df, columns=['category', 'region', 'size_bin', 'room_count_bin', 'bathroom_count_bin'], drop_first=True)

    # Align columns with train set
    for col in train_columns:
        if col not in df.columns:
            df[col] = 0
    df = df[train_columns]

    return df



In [28]:
# Predict on Test Data
test_data = pd.read_csv('/kaggle/input/predicting-estate-prices-in-tunisia/test_data_with_id.csv')
train_columns = X_train.columns
test_data_preprocessed = prepare_test_data(test_data, train_columns)
test_data_scaled = scaler.transform(test_data_preprocessed)
test_data_scaled = pd.DataFrame(test_data_scaled, columns=train_columns)
predicted_prices = gb_best.predict(test_data_scaled)

# Create Submission File
submission = pd.DataFrame({
    'ID': test_data['ID'][:len(predicted_prices)],  # Adjust if needed
    'Predicted_Price': predicted_prices
})

submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")


Submission file created: submission.csv


In [29]:
# Predict on test data for submission
test_data = pd.read_csv('/kaggle/input/predicting-estate-prices-in-tunisia/test_data_with_id.csv')
train_columns = X_train.columns
test_data_preprocessed = prepare_test_data(test_data, train_columns)
test_data_scaled = scaler.transform(test_data_preprocessed)
test_data_scaled = pd.DataFrame(test_data_scaled, columns=train_columns)

predicted_log_prices = ensemble_model.predict(test_data_scaled)
# predicted_prices = 10 ** predicted_log_prices  # Convert predictions back to the original scale

# Create Submission File
submission = pd.DataFrame({
    'ID': test_data['ID'][:len(predicted_log_prices)],
    'Predicted_Price': predicted_log_prices
})
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")


Submission file created: submission.csv


In [30]:
submission

,ID,Predicted_Price
0,8934,5.209293
1,4634,3.658331
2,7749,2.905969
3,6400,5.380343
4,10753,5.196761
...,...,...
2545,3842,2.763122
2546,7085,5.412804
2547,46,5.521911
2548,1149,2.683296
